# Custom Retriever
> Implement the `Retriever` protocol using PEP 544 structural subtyping.

| Module | `anchor.retrieval` |
|--------|--------------------|
| Key classes | `Retriever` protocol |
| Difficulty | Intermediate |

Anchor's `Retriever` is a PEP 544 `Protocol` class. Any object with matching
`retrieve()` and `index()` methods is a valid retriever -- no inheritance required.

**Time:** ~8 minutes

## Setup

In [ ]:
from anchor.models import ContextItem, SourceType, QueryBundle
from typing import Protocol, runtime_checkable

## The Retriever Protocol
Here is what the protocol looks like (simplified). Your class just needs
to implement `retrieve()` and `index()`.

In [ ]:
# This is the protocol defined by Anchor (shown for reference)
#
# @runtime_checkable
# class Retriever(Protocol):
#     def retrieve(self, query: QueryBundle, top_k: int = 10) -> list[ContextItem]: ...
#     def index(self, items: list[ContextItem]) -> None: ...

print("Retriever protocol: retrieve(query, top_k) -> list[ContextItem]")
print("                     index(items) -> None")

## Implement a Custom Retriever
We build a simple retriever that matches documents by substring overlap.

In [ ]:
class SubstringRetriever:
    """A custom retriever that scores documents by substring overlap with the query."""

    def __init__(self):
        self._items: list[ContextItem] = []

    def index(self, items: list[ContextItem]) -> None:
        """Add items to the internal store."""
        self._items.extend(items)

    def retrieve(self, query: QueryBundle, top_k: int = 10) -> list[ContextItem]:
        """Score items by word overlap and return the top-k."""
        query_words = set(query.query_str.lower().split())

        scored = []
        for item in self._items:
            doc_words = set(item.content.lower().split())
            overlap = len(query_words & doc_words)
            score = overlap / max(len(query_words), 1)
            scored.append((score, item))

        scored.sort(key=lambda x: x[0], reverse=True)

        results = []
        for score, item in scored[:top_k]:
            # Return a copy with the updated score
            results.append(ContextItem(
                id=item.id,
                content=item.content,
                source=item.source,
                score=score,
                priority=item.priority,
                token_count=item.token_count,
            ))
        return results

print("SubstringRetriever class defined.")

## Verify Protocol Conformance
Use `isinstance()` with the runtime-checkable protocol.

In [ ]:
from anchor.retrieval import Retriever

my_retriever = SubstringRetriever()

print(f"Is SubstringRetriever a Retriever? {isinstance(my_retriever, Retriever)}")

## Index and Retrieve

In [ ]:
items = [
    ContextItem(id="custom-1", content="Python is great for data science.",
               source=SourceType.RETRIEVAL, score=0.0, priority=5, token_count=8),
    ContextItem(id="custom-2", content="Rust ensures memory safety.",
               source=SourceType.RETRIEVAL, score=0.0, priority=5, token_count=6),
    ContextItem(id="custom-3", content="Data pipelines process large datasets.",
               source=SourceType.RETRIEVAL, score=0.0, priority=5, token_count=7),
    ContextItem(id="custom-4", content="Go is designed for concurrent programming.",
               source=SourceType.RETRIEVAL, score=0.0, priority=5, token_count=8),
]

my_retriever.index(items)
print(f"Indexed {len(items)} items.")

In [ ]:
query = QueryBundle(query_str="data science pipelines")
results = my_retriever.retrieve(query, top_k=3)

print(f"Query: '{query.query_str}'")
print(f"Results ({len(results)}):\n")
for i, r in enumerate(results):
    print(f"  [{i+1}] score={r.score:.4f}  {r.content}")

## Use the Custom Retriever in a RoutedRetriever
Since it satisfies the protocol, it plugs into any Anchor component
that accepts a `Retriever`.

In [ ]:
from anchor.retrieval import RoutedRetriever, KeywordRouter

router = KeywordRouter(
    routes={"custom": ["data", "science", "pipeline"]},
    default_route="custom",
)

routed = RoutedRetriever(
    router=router,
    retrievers={"custom": my_retriever},
)

results = routed.retrieve(QueryBundle(query_str="data pipeline"), top_k=2)

print("Routed through RoutedRetriever:")
for r in results:
    print(f"  {r.id}: score={r.score:.4f}  {r.content[:50]}")

## Key Takeaways
- Anchor uses PEP 544 structural subtyping -- no base class inheritance needed
- Implement `retrieve(query, top_k)` and `index(items)` to satisfy the protocol
- Use `isinstance(obj, Retriever)` to verify conformance at runtime
- Custom retrievers plug seamlessly into `RoutedRetriever`, `HybridRetriever`, etc.
- This pattern encourages composition over inheritance

**Back to:** [Retrieval README](README.md)